# الفصل 31: LoRA: التكيّف منخفض الرتبة بعمق

> LoRA هي التقنية الأكثر استخدامًا عالميًا لضبط النماذج الكبيرة اليوم.

- **الجزء:** 7 — التخصيص الفعّال والمحاذاة
- **الفصل:** 31
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 30: مقدمة إلى PEFT: التدريب الفعّال من حيث البارامترات
- **الوسوم:** LoRA، Low-Rank، Rank، Alpha

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch31-lora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch31-lora.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch31-lora.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> LoRA هي التقنية الأكثر استخدامًا عالميًا لضبط النماذج الكبيرة اليوم.

> **بيانات الفصل**
>
> - **الجزء:** 7 — التخصيص الفعّال والمحاذاة
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 30: مقدمة إلى PEFT: التدريب الفعّال من حيث البارامترات


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم عدد المعاملات: لماذا LoRA فعّال جدًا؟
- فهم أي الطبقات نستهدف بـ LoRA؟ (دراسة من ورقة QLoRA)
- فهم اختيار الرتبة `r` ومعامل `α`


## 1. لماذا يهمّك هذا الموضوع؟

LoRA هي **التقنية الأكثر استخدامًا عالميًا** لضبط النماذج الكبيرة اليوم. تُستخدم في كل مكان: من ضبط Stable Diffusion على أنماط فنية معينة، إلى ضبط Llama لمهام طبية، إلى منصات مثل Hugging Face PEFT وUnsloth وAxolotl. فهمها بعمق رياضي وكودي هو **مهارة أساسية** لأي مهندس LLM اليوم.


## 2. الخريطة الذهنية


```text
الفرضية الأساسية لـ LoRA:
"التحديث اللازم لأوزان النموذج أثناء الضبط الدقيق
 له رُتبة جوهرية منخفضة (low intrinsic rank)"

           W (مجمّد، d×k)
                +
         ΔW = B·A  (التحديث)
                │
      B (d×r)      A (r×k)
      متعلّم         متعلّم
      r << min(d,k)
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

عندما نُدرِّب طبقة خطية `W` كاملة الرتبة (full-rank)، فإن معظم "اتجاهات" التغيير الممكنة غير مُستخدَمة فعليًا؛ الأبحاث (Aghajanyan et al., 2020) أظهرت أن التحديثات الفعّالة أثناء الضبط الدقيق تقع ضمن **فضاء ذي رتبة منخفضة جدًا**. لذا بدلًا من تعلّم مصفوفة `ΔW` الكاملة (d×k معامل)، نُقيّدها بالشكل `ΔW = B·A` حيث `B` بحجم `d×r` و`A` بحجم `r×k`، و`r` صغير جدًا (4، 8، 16، 32، 64).

### 🟡 طبقة الآلية

الصيغة الرياضية الكاملة لطبقة LoRA:


```text
h = W₀·x + ΔW·x = W₀·x + (α/r)·B·A·x
```


حيث:
- `W₀`: الأوزان الأصلية **المجمّدة بالكامل** (لا تتلقى تدرجات).
- `B`: مصفوفة بحجم `(d_out × r)`، **تُهيَّأ بالصفر**.
- `A`: مصفوفة بحجم `(r × d_in)`، **تُهيَّأ عشوائيًا** (Gaussian/Kaiming).
- `α`: معامل تحجيم ثابت (LoRA scaling factor).
- `r`: الرتبة (rank)، أهم مُفرط-معامل (hyperparameter) في LoRA.

> 🔴 **لماذا B تُهيَّأ بالصفر وA عشوائيًا (وليس العكس)؟**
> عند `B=0`، فإن `ΔW = B·A = 0` بغض النظر عن قيمة `A`، لذا يبدأ النموذج من حالته الأصلية تمامًا (مطابق لمبدأ near-identity init في الفصل السابق). لو هيّأنا `A=0` بدلًا من ذلك، لكانت التدرجات بالنسبة لـ `B` معدومة في البداية (لأن `∂L/∂B = ∂L/∂h · xᵀAᵀ` يعتمد على `A`)، فتتوقف عملية التعلّم بالكامل قبل أن تبدأ.

### عدد المعاملات: لماذا LoRA فعّال جدًا؟

لطبقة خطية بحجم `d×k` (لنفترض d=k=4096 كما في Llama):


```text
Full Fine-Tuning:  d × k = 4096 × 4096 = 16,777,216 معامل
LoRA (r=8):        d×r + r×k = 4096×8 + 8×4096 = 65,536 معامل

نسبة التخفيض: 65,536 / 16,777,216 ≈ 0.39%  (أي ~256× أقل)
```


### 🔴 طبقة التعمّق — تطبيق LoRA من الصفر بالكامل


In [ ]:
import torch
import torch.nn as nn
import math

class LoRALayer(nn.Module):
    """
    تطبيق LoRA من الصفر طبقًا لورقة Hu et al. (2021).
    يُستخدم كطبقة "إضافية" بجانب أي nn.Linear مجمّدة.
    """
    def __init__(self, in_features: int, out_features: int,
                 rank: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank  # معامل التحجيم α/r

        # A: تهيئة Kaiming uniform (كما في الورقة الأصلية)
        self.lora_A = nn.Parameter(torch.zeros(rank, in_features))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        # B: تهيئة بالصفر (حرج جدًا لضمان ΔW=0 في البداية)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., in_features)
        x = self.dropout(x)
        # ΔW·x = B·A·x ، لاحظ ترتيب العمليات لتوفير الحساب: (x·Aᵀ)·Bᵀ
        result = x @ self.lora_A.T  # (..., rank)
        result = result @ self.lora_B.T  # (..., out_features)
        return result * self.scaling

class LinearWithLoRA(nn.Module):
    """
    يُغلّف nn.Linear أصلية (مجمّدة) بطبقة LoRA مُضافة.
    هذا هو النمط الفعلي المُستخدَم في مكتبة PEFT من Hugging Face.
    """
    def __init__(self, base_linear: nn.Linear, rank: int = 8,
                 alpha: int = 16, dropout: float = 0.05):
        super().__init__()
        self.base_linear = base_linear
        # تجميد الطبقة الأصلية بالكامل — هذا هو جوهر LoRA
        for param in self.base_linear.parameters():
            param.requires_grad = False

        self.lora = LoRALayer(
            in_features=base_linear.in_features,
            out_features=base_linear.out_features,
            rank=rank, alpha=alpha, dropout=dropout,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_linear(x)       # W₀·x  (بدون تدرجات)
        lora_out = self.lora(x)               # (α/r)·B·A·x  (بتدرجات)
        return base_out + lora_out

    def merge_weights(self) -> nn.Linear:
        """
        دمج أوزان LoRA في الطبقة الأصلية لإنتاج طبقة خطية واحدة
        بدون أي حمل حسابي إضافي أثناء الاستدلال (inference overhead = صفر).
        هذه هي الميزة الحاسمة لـ LoRA مقارنة بـ Adapter Layers.
        """
        merged = nn.Linear(self.base_linear.in_features,
                            self.base_linear.out_features,
                            bias=self.base_linear.bias is not None)
        with torch.no_grad():
            delta_w = (self.lora.lora_B @ self.lora.lora_A) * self.lora.scaling
            merged.weight.copy_(self.base_linear.weight + delta_w)
            if self.base_linear.bias is not None:
                merged.bias.copy_(self.base_linear.bias)
        return merged

def apply_lora_to_model(model: nn.Module, target_modules: list,
                         rank: int = 8, alpha: int = 16,
                         dropout: float = 0.05) -> nn.Module:
    """
    تطبيق LoRA تلقائيًا على كل الطبقات التي تطابق أسماء target_modules
    (نمط شائع: ["q_proj", "k_proj", "v_proj", "o_proj"] لنماذج Llama).
    """
    for name, module in model.named_modules():
        for target in target_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parent_name = ".".join(name.split(".")[:-1])
                child_name = name.split(".")[-1]
                parent = model.get_submodule(parent_name) if parent_name else model
                setattr(parent, child_name,
                        LinearWithLoRA(module, rank=rank, alpha=alpha, dropout=dropout))

    # تجميد كل ما تبقى من النموذج (embeddings, norms, إلخ) باستثناء LoRA
    for name, param in model.named_parameters():
        if "lora" not in name:
            param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"LoRA مُطبّق: {trainable:,} / {total:,} معامل قابل للتدريب "
          f"({100*trainable/total:.4f}%)")
    return model


### أي الطبقات نستهدف بـ LoRA؟ (دراسة من ورقة QLoRA)

| إعداد الاستهداف | عدد المعاملات القابلة للتدريب | الأداء (نسبة إلى Full FT) |
|---|---|---|
| `q_proj, v_proj` فقط (الإعداد الأصلي في ورقة LoRA) | الأقل | جيد |
| `q_proj, k_proj, v_proj, o_proj` | متوسط | أفضل |
| **كل الطبقات الخطية** (`q,k,v,o,gate,up,down`) | الأعلى (لكن لا يزال <1%) | **الأفضل — موصى به حاليًا** |

> 💡 وجدت ورقة QLoRA (Dettmers et al., 2023) تجريبيًا أن استهداف **كل** الطبقات الخطية في المحوّل (وليس فقط Attention) يُحقق أفضل أداء إجمالي، وأصبح هذا هو المعيار العالمي الحالي.

### اختيار الرتبة `r` ومعامل `α`


In [ ]:
def recommend_lora_config(task_complexity: str, model_size_billions: float):
    """
    توصيات عملية لاختيار r وα بناءً على تعقيد المهمة وحجم النموذج،
    مُستخلصة من الممارسات الشائعة في المجتمع (Hugging Face, Unsloth).
    """
    configs = {
        "simple_classification": {"r": 4, "alpha": 8},
        "domain_adaptation": {"r": 16, "alpha": 32},
        "complex_instruction_following": {"r": 32, "alpha": 64},
        "full_capability_transfer": {"r": 64, "alpha": 128},
    }
    config = configs.get(task_complexity, {"r": 16, "alpha": 32})

    # قاعدة عملية شائعة: alpha = 2×r كبداية، ثم ضبط تجريبي
    print(f"التوصية لمهمة '{task_complexity}' على نموذج {model_size_billions}B:")
    print(f"  rank (r) = {config['r']}")
    print(f"  alpha (α) = {config['alpha']}")
    print(f"  نسبة α/r = {config['alpha']/config['r']}")
    return config


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: درّب LoRA (r=8) على طبقة خطية بحجم `(4096, 4096)` لتعلّم تحويل بسيط، وتحقّق من أن دمج الأوزان ينتج سلوكًا مطابقًا.


In [ ]:
torch.manual_seed(42)

# إنشاء الطبقة الأصلية (تمثيل مبسّط لـ q_proj في محوّل)
base_layer = nn.Linear(4096, 4096, bias=False)
lora_layer = LinearWithLoRA(base_layer, rank=8, alpha=16)

# تدريب تجريبي قصير: تعليم LoRA لتقريب تحويل مستهدف معيّن
target_transform = nn.Linear(4096, 4096, bias=False)
optimizer = torch.optim.AdamW(
    [p for p in lora_layer.parameters() if p.requires_grad], lr=1e-3
)

x_sample = torch.randn(32, 4096)
for step in range(200):
    optimizer.zero_grad()
    pred = lora_layer(x_sample)
    target = target_transform(x_sample)
    loss = nn.functional.mse_loss(pred, target)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"Step {step}: loss = {loss.item():.6f}")

# التحقق من تطابق الدمج
merged_layer = lora_layer.merge_weights()
with torch.no_grad():
    out_unmerged = lora_layer(x_sample)
    out_merged = merged_layer(x_sample)
    max_diff = (out_unmerged - out_merged).abs().max().item()
    print(f"\nأقصى فرق بين النموذج قبل/بعد الدمج: {max_diff:.2e}")
    # يجب أن يكون قريبًا جدًا من الصفر (خطأ عددي فقط)


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| استهدف كل الطبقات الخطية | وليس فقط Attention، وفق توصيات ورقة QLoRA |
| LR أعلى من Full FT | عادة 1e-4 إلى 3e-4 للـ LoRA (مقارنة بـ 1e-5 إلى 5e-5 للـ Full FT) |
| ابدأ بـ r=16 كخط أساس | وزِد فقط إذا رأيت underfitting واضح على مجموعة التحقق |
| استخدم Dropout في LoRA | 0.05-0.1 يساعد في منع الإفراط في التخصيص (overfitting) خصوصًا على بيانات قليلة |
| ادمج الأوزان قبل النشر | `merge_weights()` يُزيل أي overhead حسابي أثناء الاستدلال |


## 6. أخطاء شائعة ومصائد

- **تجاهل معامل α/r**: كثيرون يتجاهلون التحجيم فيؤدي لعدم استقرار إذا تغيّرت r دون تعديل α تناسبيًا.
- **تدريب Embeddings مع LoRA في نفس الوقت دون حذر**: إذا أضفت مفردات جديدة (Tokens جديدة)، يجب معالجة embedding matrix بعناية خاصة (غالبًا عبر `modules_to_save` في مكتبة PEFT).
- **الخلط بين "الدمج" و"عدم الدمج" في الإنتاج**: الاحتفاظ بـ LoRA منفصلًا (غير مدموج) مفيد فقط إذا أردت التبديل السريع بين عدة LoRAs على نفس النموذج الأساسي (مثل S-LoRA في الخدمة الإنتاجية)، وإلا الدمج هو الخيار الأمثل للأداء.


## 7. جدول مقارنة

| التقنية | الفكرة الجوهرية | ميزة إضافية |
|---|---|---|
| **LoRA** (الأساس) | `ΔW = B·A` | بسيط، قابل للدمج |
| **DoRA** (2024) | تفكيك الوزن إلى Magnitude + Direction، LoRA يُطبَّق على الاتجاه فقط | أداء أعلى بثبات مع نفس عدد المعاملات |
| **LoRA+** (2024) | معدلات تعلّم مختلفة لـ A وB (LR أعلى لـ B) | تقارب أسرع، استغلال أفضل لسعة B |
| **VeRA** | A وB عشوائيتان ومُجمَّدتان، نتعلّم فقط متجهي تحجيم صغيرين | معاملات أقل بـ ~10× من LoRA |
| **AdaLoRA** | تخصيص الرتبة r ديناميكيًا لكل طبقة حسب أهميتها | استغلال أمثل لميزانية المعاملات |


## 8. ملخص أفضل الممارسات

- استهدف كل الطبقات الخطية الرئيسية في المحوّل.
- ابدأ بـ r=16، α=32، dropout=0.05 كإعداد افتراضي معقول.
- استخدم LR أعلى من Full FT (عادة 1e-4 إلى 3e-4).
- ادمج الأوزان دائمًا قبل النشر الإنتاجي النهائي ما لم تحتج خدمة متعددة LoRAs.
- راقب فقدان القدرات العامة (Catastrophic Forgetting) بنفس طرق الفصل 24.


## 9. أسئلة مقابلات

**س: لماذا لا تُسبّب LoRA أي بُطء في الاستدلال بعد النشر، بخلاف Adapter Layers؟**

**ج**: لأن تحديث LoRA خطي بالكامل (`ΔW = B·A`)، يمكن دمجه رياضيًا مع الأوزان الأصلية بجمع بسيط (`W_merged = W₀ + (α/r)·B·A`) **قبل** الاستدلال، لنحصل على مصفوفة واحدة بنفس أبعاد وبنية الطبقة الأصلية تمامًا. أما Adapter Layers فتحتوي على دالة تفعيل غير خطية (مثل GELU) بين `W_down` و`W_up`، مما يمنع دمجها رياضيًا في طبقة واحدة، فتبقى كطبقة منفصلة تُضيف حسابًا فعليًا (وبالتالي زمن استجابة أعلى) في كل عملية استدلال.

**س: ماذا يحدث لو اخترت rank كبيرًا جدًا (مثلًا r=512) على نموذج صغير؟**

**ج**: نظريًا يقترب LoRA من Full Fine-Tuning كلما زادت r (عند `r = min(d,k)` يصبح `B·A` قادرًا نظريًا على تمثيل أي مصفوفة `ΔW` بالكامل). لكن عمليًا هذا يُلغي الفائدة الأساسية من LoRA (الكفاءة)، وقد يزيد من خطر الإفراط في التخصيص (overfitting) خصوصًا مع بيانات ضبط دقيق قليلة، لأن الفرضية الأساسية لـ LoRA (أن `ΔW` له رتبة جوهرية منخفضة) تصبح غير مُستغَلة فعليًا.


## 10. تمرين عملي

طبّق `apply_lora_to_model` على نموذج `GPT-2 small` من Hugging Face، درّبه على توليد نصوص عربية بأسلوب شعري محدد، ثم:
1. قارن حجم ملف الـ checkpoint (LoRA فقط مقابل النموذج الكامل).
2. ادمج الأوزان واختبر تطابق المخرجات قبل/بعد الدمج.
3. جرّب rank مختلفة (4, 16, 64) وقارن جودة التوليد.


## 11. مراجع للتعمّق

- Hu et al. (2021) — *LoRA: Low-Rank Adaptation of Large Language Models*
- Aghajanyan et al. (2020) — *Intrinsic Dimensionality Explains the Effectiveness of LMs Fine-Tuning*
- Liu et al. (2024) — *DoRA: Weight-Decomposed Low-Rank Adaptation*
- Hayou et al. (2024) — *LoRA+: Efficient Low Rank Adaptation of Large Models*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التخصيص الفعّال والمحاذاة / الفصل 31: LoRA: التكيّف منخفض الرتبة بعمق.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
